# Automated Program Repair via Prompt Learning: SWE Benchmark Dataset 

## 0. Environment & Setup

In [1]:
%pip install groq
%pip install datasets

In [2]:
import os
import pandas as pd
import json
import difflib
from string import Template
from datetime import datetime
from textwrap import dedent
import re
from difflib import unified_diff
from typing import Optional
from difflib import SequenceMatcher
import shutil
import subprocess
import tempfile
from difflib import SequenceMatcher, unified_diff
from typing import Optional, Dict, Any
import matplotlib.pyplot as plt
import csv
import sys
from groq import Groq
from datasets import load_dataset
from typing import Optional, Dict, Any, List, Tuple
import time
import difflib
import ast
import re
from pathlib import Path

In [3]:
# Load .env files for API Key
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
USE_GROQ = True      

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')

if USE_GROQ and not GROQ_API_KEY:
    print('WARNING: GROQ_API_KEY not set. Set env var GROQ_API_KEY to enable Groq backend.')

In [5]:
client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
print("Groq client initialized")

Groq client initialized


In [6]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one word."}
    ],
    temperature=0.2,
)

print(resp.choices[0].message.content)

Hello.


## 1. SWE Functions

### A. Dataset Loading

In [9]:
# SWE-bench dataset loading 
HF_DATASET_NAME = "princeton-nlp/SWE-bench_Lite"
HF_SPLIT = "test"

# Local directories
ROOT_DIR = Path(".")
REPOS_DIR = ROOT_DIR / "swebench_repos"
ARTIFACT_DIR = ROOT_DIR / "artifacts_swebench"
REPOS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Set Parameters
DEFAULT_LIMIT = 300
MAX_CONTEXT_CHARS = 14000
MAX_FILE_CHARS = 2000
MAX_FILES = 6

In [10]:
ds = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

In [11]:
print(len(ds))
print(ds[0].keys())
print(ds[0]["instance_id"])
print(ds[0]["repo"])
print(ds[0]["base_commit"])
print(ds[0]["problem_statement"])

300
dict_keys(['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit'])
astropy__astropy-12907
astropy/astropy
d16bfe05a744909de4b27f5875fe0d4ed41ce607
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix 

### B. Repo Checkout

In [13]:
def safe_repo_dir_name(repo: str) -> str:
    return repo.replace("/", "__")


def run_cmd(cmd: List[str], cwd: Optional[Path] = None) -> subprocess.CompletedProcess:
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        check=True,
    )


def checkout_repo(repo: str, base_commit: str, repos_dir: Path = REPOS_DIR) -> Path:
    local_dir = repos_dir / safe_repo_dir_name(repo)

    if not local_dir.exists():
        repo_url = f"https://github.com/{repo}.git"
        print(f"[clone] {repo_url}")
        run_cmd(["git", "clone", repo_url, str(local_dir)])

    print(f"[fetch] {repo}")
    run_cmd(["git", "fetch", "--all"], cwd=local_dir)

    print(f"[checkout] {repo} @ {base_commit}")
    run_cmd(["git", "checkout", "--force", base_commit], cwd=local_dir)

    return local_dir

### C. Configuration

### D. Function Retrieval

In [16]:
## LLM Localization
from llm_localization import localize_instance
from llm_localization import find_files_named_in_issue

In [17]:
def tokenize_issue_text(text: str) -> List[str]:
    words = re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", text.lower())
    stop = {
        "the", "and", "for", "with", "that", "this", "from", "when", "where",
        "into", "have", "has", "had", "issue", "error", "fails", "failure",
        "should", "would", "could", "there", "their", "about", "while",
        "after", "before", "being", "return", "returns", "using", "used",
        "fix", "bug", "problem", "incorrect", "unexpected", "wrong"
    }

    tokens = [w for w in words if w not in stop]

    seen = set()
    deduped = []
    for t in tokens:
        if t not in seen:
            deduped.append(t)
            seen.add(t)

    return deduped[:40]


def is_identifier_like(token: str) -> bool:
    return (
        "_" in token
        or any(ch.isdigit() for ch in token)
        or len(token) >= 8
    )


def is_code_file(path: Path) -> bool:
    allowed = {
        ".py", ".pyi", ".js", ".ts", ".tsx", ".jsx", ".java",
        ".go", ".rb", ".rs", ".php", ".c", ".cc", ".cpp", ".h", ".hpp"
    }
    return path.suffix.lower() in allowed


def score_file(path: Path, text: str, keywords: List[str]) -> int:
    score = 0
    lower_path = str(path).lower()
    lower_text = text.lower()

    path_name = Path(lower_path).name

    for kw in keywords:
        kw_score = 2 if is_identifier_like(kw) else 1

        path_hits = lower_path.count(kw)
        body_hits = lower_text.count(kw)

        score += path_hits * (5 * kw_score)
        if kw in path_name:
            score += 4 * kw_score

        score += min(body_hits, 10) * kw_score

    if "test" in lower_path or "tests" in lower_path:
        score += 2

    return score


def best_snippet_window(text: str, keywords: List[str], max_file_chars: int) -> str:
    lower_text = text.lower()
    hit_positions = []

    for kw in keywords:
        start = 0
        while True:
            pos = lower_text.find(kw, start)
            if pos == -1:
                break
            weight = 2 if is_identifier_like(kw) else 1
            hit_positions.append((pos, weight))
            start = pos + len(kw)

    if not hit_positions:
        return text[:max_file_chars]

    best_start = 0
    best_score = -1

    for pos, _ in hit_positions:
        window_start = max(0, pos - max_file_chars // 2)
        window_end = min(len(text), window_start + max_file_chars)

        window_score = 0
        for other_pos, weight in hit_positions:
            if window_start <= other_pos < window_end:
                window_score += weight

        if window_score > best_score:
            best_score = window_score
            best_start = window_start

    best_end = min(len(text), best_start + max_file_chars)
    return text[best_start:best_end]


def retrieve_context_for_instance(
    instance: Dict[str, Any],
    repos_dir: Path = REPOS_DIR,
    max_context_chars: int = MAX_CONTEXT_CHARS,
    max_file_chars: int = MAX_FILE_CHARS,
    max_files: int = MAX_FILES,
) -> Tuple[str, Dict[str, str]]:
    repo_path = checkout_repo(instance["repo"], instance["base_commit"], repos_dir=repos_dir)
    keywords = tokenize_issue_text(instance["problem_statement"])

    scored_files = []
    original_contents = {}

    for path in repo_path.rglob("*"):
        if not path.is_file():
            continue
        if ".git" in path.parts:
            continue
        if not is_code_file(path):
            continue

        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        rel_path = path.relative_to(repo_path)
        score = score_file(rel_path, text, keywords)

        if score > 0:
            scored_files.append((score, path, text))

    scored_files.sort(key=lambda x: x[0], reverse=True)

    chunks = []
    used = 0

    for score, path, text in scored_files[:max_files]:
        rel = path.relative_to(repo_path).as_posix()
        original_contents[rel] = text

        snippet = best_snippet_window(text, keywords, max_file_chars)

        block = f"\n# TARGET FILE: {rel}\n{snippet}\n"

        if used + len(block) > max_context_chars:
            break

        chunks.append(block)
        used += len(block)

    return "\n".join(chunks), original_contents

## 2. MSR Functions

### A. Prompt Template:

In [20]:
REPAIR_TEMPLATE_SWE = Template(r"""
You are a security reviewer fixing a real software engineering issue in a GitHub repository. Fix the following function to resolve the issue.
- Issue description: $desc

You are given a target function and some additional repository context.

Instructions:
- You must edit ONLY the target function.
- Use the additional context ONLY if it is relevant.
- Return ONLY the corrected Python function.
- Keep the SAME function name: $func_name
- Keep the SAME function signature unless the bug explicitly requires changing it.
- Keep behavior identical except for the fix.
- Do NOT return any other function.
- Do NOT include markdown fences.
- Do NOT include explanations or extra text.
- If the issue description does not affect this function, return the function unchanged.
- Make the minimal necessary change.

Target  Function (to fix):
$func

Additional Context (may be useful):
$retrieved_context
""".strip())

def build_repair_prompt_swe(
    vuln_func: str,
    desc: str | None = None,
    func_name: str | None = None,
    retrieved_context: str | None = None
):
    return REPAIR_TEMPLATE_SWE.substitute(
        desc=desc or "No description provided.",
        func=vuln_func.strip(),
        func_name=func_name or "UNKNOWN_FUNCTION",
        retrieved_context=retrieved_context or "No additional context provided."
    )

In [21]:
def normalize_model_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_+-]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    m = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    text = re.sub(
        r"^\s*(here(?:'s| is)\s+the\s+(?:fixed|corrected)\s+(?:function|code)\s*:?)",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    return text

### B. LLM Backend Setup

In [23]:
def run_groq(prompt: str, model: str = "llama-3.3-70b-versatile") -> Optional[str]:

    MAX_PROMPT_CHARS = 20000
    
    if not GROQ_API_KEY:
        print("Groq backend disabled or missing API key.")
        return None

    if len(prompt) > MAX_PROMPT_CHARS:
        print("Prompt too large, truncating...")
        prompt = prompt[:MAX_PROMPT_CHARS]
    
    try:
        from groq import Groq

        client = Groq(api_key=GROQ_API_KEY)

        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a careful C security patching assistant."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
        )

        return resp.choices[0].message.content

    except Exception as e:
        print("Groq error:", e)
        return None


### C. Evaluation 

In [25]:
CHATTER_PATTERNS = [
    r"^\s*here(?:'s| is)\b",
    r"^\s*the fixed function\b",
    r"^\s*the corrected code\b",
    r"^\s*explanation\b",
    r"^\s*sure[,:\s]",
    r"^\s*i fixed\b",
]

def quick_heuristics(original_code: str, patched_code: str) -> Dict[str, Any]:
    """
    Lightweight sanity checks for SWE-bench style bug-fix patches.
    """
    original_code = original_code or ""
    patched_code = patched_code or ""

    result: Dict[str, Any] = {}

    orig_len = len(original_code)
    patched_len = len(patched_code)

    orig_lines = original_code.splitlines()
    patched_lines = patched_code.splitlines()

    # Basic checks
    result["non_empty_patch"] = bool(patched_code.strip())
    result["same_text"] = (original_code == patched_code)
    result["orig_chars"] = orig_len
    result["patched_chars"] = patched_len
    result["orig_lines"] = len(orig_lines)
    result["patched_lines"] = len(patched_lines)
    result["char_ratio"] = patched_len / max(1, orig_len)

    # Size sanity
    result["too_large"] = patched_len > 3 * max(1, orig_len)
    result["too_small"] = patched_len < 0.3 * max(1, orig_len)

    # Output contamination checks
    result["has_markdown_fences"] = ("```" in patched_code)
    result["has_xml_tags"] = bool(re.search(r"</?(function|code|patch|response)>", patched_code))
    result["has_chatty_prefix"] = any(
        re.search(pattern, patched_code, flags=re.IGNORECASE | re.MULTILINE)
        for pattern in CHATTER_PATTERNS
    )

    try:
        ast.parse(patched_code)
        result["syntax_valid"] = True
        result["syntax_error"] = None
    except SyntaxError as e:
        result["syntax_valid"] = False
        result["syntax_error"] = f"{e.msg} (line {e.lineno})"
    except Exception as e:
        result["syntax_valid"] = False
        result["syntax_error"] = str(e)

    # Structural checks
    stripped = patched_code.strip()
    result["looks_like_python_code"] = bool(
        re.search(r"\b(def|class|if|for|while|try|return|import|from)\b", stripped)
    )

    # Deletion-heavy / rewrite-heavy heuristics
    removed_line_fraction = 0.0
    if len(orig_lines) > 0:
        removed_line_fraction = max(0, (len(orig_lines) - len(patched_lines))) / len(orig_lines)
    result["deletes_too_much"] = removed_line_fraction > 0.7

    # Function header presence (regards to original code)
    orig_def_match = re.search(r"^\s*def\s+([A-Za-z_]\w*)\s*\(", original_code, flags=re.MULTILINE)
    if orig_def_match:
        fn_name = orig_def_match.group(1)
        result["original_function_name"] = fn_name
        result["function_name_preserved"] = bool(
            re.search(rf"^\s*def\s+{re.escape(fn_name)}\s*\(", patched_code, flags=re.MULTILINE)
        )
    else:
        result["original_function_name"] = None
        result["function_name_preserved"] = None

    # Scoring
    score = 0.0

    if result["non_empty_patch"]:
        score += 1.0
    else:
        score -= 3.0

    if not result["same_text"]:
        score += 1.0
    else:
        score -= 1.5

    if result["syntax_valid"]:
        score += 2.0
    else:
        score -= 3.0

    if not result["too_large"]:
        score += 0.5
    else:
        score -= 2.0

    if not result["too_small"]:
        score += 0.25
    else:
        score -= 1.0

    if not result["has_markdown_fences"]:
        score += 0.5
    else:
        score -= 1.5

    if not result["has_xml_tags"]:
        score += 0.25
    else:
        score -= 0.75

    if not result["has_chatty_prefix"]:
        score += 0.5
    else:
        score -= 1.5

    if result["looks_like_python_code"]:
        score += 0.5
    else:
        score -= 1.0

    if not result["deletes_too_much"]:
        score += 0.5
    else:
        score -= 1.5

    if result["function_name_preserved"] is True:
        score += 0.5
    elif result["function_name_preserved"] is False:
        score -= 1.5

    result["score_heuristic"] = round(score, 3)
    return result

# LLM Evaluator
def build_eval_prompt_swe(original_func: str, patched_func: str, problem_desc: str, target_func_name: str) -> str:
    return dedent(f"""
    You are a strict Python code reviewer evaluating a proposed bug-fix patch.
    
    TARGET FUNCTION NAME:
    {target_func_name}
    
    ISSUE DESCRIPTION:
    {problem_desc}
    
    ORIGINAL FUNCTION (vulnerable/buggy):
    ```python
    {original_func}
    ```

    PATCHED FUNCTION:
    ```diff
    {patched_func}
    ```
        
    Evaluate the patch using these rules in order:

    1. Function identity check:
       - The patched code must still define the same target function: `{target_func_name}`.
       - If the patch changes the function name, replaces it with a different function, or does not clearly patch `{target_func_name}`, mark it INCORRECT.

    2. Scope check:
       - The patch must remain a modification of the same function, not a rewrite of unrelated logic.
       - If the patch appears unrelated to the target function or issue, mark it INCORRECT.

    3. Relevance check:
       - The patch should directly address the issue description.
       - If the change appears generic, unrelated, or only weakly connected to the issue, prefer INCORRECT over UNCLEAR.

    4. Minimality check:
       - The patch should make only the minimal necessary change.
       - Large rewrites, unnecessary restructuring, or added unrelated logic should lower confidence and minimality.
       - Reject patches that change indentation, move functions out of classes, change unrelated logic or introduce new variables.

    5. Correctness judgment:
       - CORRECT: strong evidence the patch preserves function identity, is relevant to the issue, and makes a plausible minimal fix.
       - UNCLEAR: the patch is relevant and preserves identity, but there is not enough evidence to confidently judge correctness.
       - INCORRECT: the patch changes function identity, changes scope, appears unrelated, or is unlikely to fix the described issue.

    Important:
    - Be conservative.
    - Do not reward patches just because they are syntactically valid.
    - Do not assume the patch is correct unless the connection to the issue is clear.
    - If the patch looks plausible but the link to the described bug is weak, return UNCLEAR or INCORRECT.
    - If there is not strong evidence that the patch is a correct and relevant fix for the target function, do NOT mark it CORRECT.
    - Reject patches that change indentation or sc
    
    Output ONLY the JSON verdict:
    {{
      "verdict": "CORRECT" | "UNCLEAR" | "INCORRECT",
      "confidence": 0.0,
      "reason": "short explanation",
      "minimality": "HIGH" | "MEDIUM" | "LOW"
    }}
    """).strip()

# LLM Scoring function
def score_llm_eval(llm_eval_json: Optional[dict]) -> float:
    if not llm_eval_json:
        return 0.0
    
    verdict = llm_eval_json.get("verdict", "UNCLEAR")
    minimality = str(llm_eval_json.get("minimality", "MEDIUM")).upper()
    conf = llm_eval_json.get("confidence", 0.0)
    
    unsafe_paths = llm_eval_json.get("unsafe_paths", [])
    remaining_risks = llm_eval_json.get("remaining_risks", [])
    
    try:
        conf = float(conf)
    except Exception:
        conf = 0.0
    
    # Confidence
    conf = max(0.0, min(1.0, conf))
    
    # Penalty for remaining issues
    penalty = (len(unsafe_paths) * 0.3 + len(remaining_risks) * 0.2) * conf
    
    # Base score using MSR logic
    if verdict == "CORRECT":
        base_score = 2.0 * conf
    elif verdict == "INCORRECT":
        base_score = -2.0 * conf
    else:  # UNCLEAR
        base_score = 0.5 * conf
    
    # Minimality adjustment
    if minimality == "HIGH":
        min_adj = 0.3 * conf
    elif minimality == "LOW":
        min_adj = -0.3 * conf
    else: 
        min_adj = 0.0

    return base_score + min_adj

def diff_stats(original: str, patched: str) -> Dict[str, Any]:
    original = original or ""
    patched = patched or ""

    orig_lines = original.splitlines()
    patched_lines = patched.splitlines()

    matcher = difflib.SequenceMatcher(None, orig_lines, patched_lines)

    added_lines = 0
    removed_lines = 0
    changed_lines = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "insert":
            added_lines += (j2 - j1)
        elif tag == "delete":
            removed_lines += (i2 - i1)
        elif tag == "replace":
            removed = i2 - i1
            added = j2 - j1
            removed_lines += removed
            added_lines += added
            changed_lines += max(removed, added)

    orig_len = len(original)
    patched_len = len(patched)

    return {
        "same_text": original == patched,
        "orig_chars": orig_len,
        "patched_chars": patched_len,
        "orig_lines": len(orig_lines),
        "patched_lines": len(patched_lines),
        "char_delta": patched_len - orig_len,
        "line_delta": len(patched_lines) - len(orig_lines),
        "added_lines": added_lines,
        "removed_lines": removed_lines,
        "changed_lines": changed_lines,
        "edit_ratio": 1.0 - matcher.ratio(),
        "size_ratio": (patched_len / max(1, orig_len)),
    }

def evaluate_patch_swe(
    original_func: str,
    patched_func: str,
    problem_desc: str,
    target_func_name: str,
    backend: str = "groq",
    model: str = "llama-3.1-8b-instant",
) -> Dict[str, Any]:
    
    original_func = original_func or ""
    patched_func = patched_func or ""
    problem_desc = problem_desc or "No description provided."

    # 1) SWE-bench patch sanity checks
    qe = quick_heuristics(original_func, patched_func)

    # 2) Edit statistics
    edits = diff_stats(original_func, patched_func)

    heuristic_score = float(qe.get("score_heuristic", 0.0))

    # Handle empty patch
    if not patched_func.strip():
        return {
            "quick_eval": qe,
            "edit_stats": edits,
            "eval_prompt": None,
            "llm_eval_raw": None,
            "llm_eval": None,
            "llm_score": 0.0,
            "final_score": heuristic_score,
        }

    # Handle obviously bad patch
    if (
        qe.get("too_large")
        or not qe.get("syntax_valid", False)
        or qe.get("has_markdown_fences", False)
        or qe.get("has_chatty_prefix", False)
    ):
        return {
            "quick_eval": qe,
            "edit_stats": edits,
            "eval_prompt": None,
            "llm_eval_raw": None,
            "llm_eval": {"verdict": "UNCLEAR", "confidence": 0.0},
            "llm_score": 0.0,
            "final_score": heuristic_score,
        }

    # 3) LLM evaluation
    eval_prompt = build_eval_prompt_swe(
        original_func=original_func,
        patched_func=patched_func,
        problem_desc=problem_desc,
        target_func_name=target_func_name
    )

    if backend == "groq":
        eval_raw = run_groq(eval_prompt, model=model)
    else:
        raise ValueError(f"Unknown backend: {backend}")

    eval_json = try_parse_json(eval_raw) if eval_raw else None

    # 4) Score LLM review
    llm_score = score_llm_eval(eval_json)

    # 5) Combine scores
    final_score = heuristic_score + llm_score

    return {
        "quick_eval": qe,
        "edit_stats": edits,
        "eval_prompt": eval_prompt,
        "llm_eval_raw": eval_raw,
        "llm_eval": eval_json,
        "llm_score": llm_score,
        "final_score": final_score,
    }

## 3. Function Extraction 

In [27]:
def extract_python_functions(file_content: str) -> List[Dict[str, Any]]:
    functions = []
    
    try:
        tree = ast.parse(file_content)
        
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                # Get the source lines for this function
                start_line = node.lineno - 1  # 0-indexed
                end_line = node.end_lineno if hasattr(node, 'end_lineno') else None
                
                if end_line:
                    lines = file_content.splitlines()
                    func_lines = lines[start_line:end_line]
                    func_content = '\n'.join(func_lines)
                    
                    functions.append({
                        'name': node.name,
                        'content': func_content,
                        'start_line': start_line,
                        'end_line': end_line,
                        'ast_node': node
                    })
    except SyntaxError as e:
        print(f"Syntax error parsing file: {e}")
        # Fallback to regex 
        functions = extract_functions_fallback(file_content)
    
    return functions

def extract_functions_fallback(file_content: str) -> List[Dict[str, Any]]:
    functions = []
    
    # Simple pattern for function definitions
    pattern = r'def\s+(\w+)\s*\([^)]*\)\s*:(.*?)(?=\ndef\s+|\Z)'
    matches = re.finditer(pattern, file_content, re.DOTALL)
    
    for match in matches:
        functions.append({
            'name': match.group(1),
            'content': match.group(0),
            'start': match.start(),
            'end': match.end()
        })
    
    return functions

def extract_c_functions(file_content: str) -> List[Dict[str, Any]]:
    functions = []
    
    # Match function patterns
    pattern = r'([a-zA-Z_][a-zA-Z0-9_]*\s+)?[a-zA-Z_][a-zA-Z0-9_]*\s*\([^;{]*\)\s*\{[^}]*\}'
    matches = re.finditer(pattern, file_content, re.DOTALL)
    
    for match in matches:
        functions.append({
            'content': match.group(0),
            'start': match.start(),
            'end': match.end()
        })
    
    return functions

def function_relevant_to_issue(func_content: str, problem_desc: str) -> bool:
    """
    Very lightweight relevance filter based on keyword overlap.
    """

    func_text = func_content.lower()
    desc_text = problem_desc.lower()

    # Extract keywords from the issue description
    keywords = set(re.findall(r"[a-zA-Z_]{4,}", desc_text))

    matches = sum(1 for k in keywords if k in func_text)

    return matches >= 2

## 4. Replace Extracted Function with Patch

In [29]:
def replace_function_in_file(original_content, old_func_content, new_func_content, func_position):
    lines = original_content.splitlines(keepends=True)

    if "start_line" in func_position and "end_line" in func_position:
        start = func_position["start_line"]
        end = func_position["end_line"]

        new_lines = new_func_content.splitlines(keepends=True)

        if new_lines and not new_lines[-1].endswith("\n"):
            new_lines[-1] += "\n"

        return "".join(lines[:start] + new_lines + lines[end:])

    if "start" in func_position and "end" in func_position:
        start = func_position["start"]
        end = func_position["end"]

        replacement = new_func_content
        if replacement and not replacement.endswith("\n"):
            replacement += "\n"

        return original_content[:start] + replacement + original_content[end:]

    return original_content.replace(old_func_content, new_func_content, 1)

def create_diff_from_replacement(original_content: str, patched_content: str, file_path: str) -> str:
    import difflib
    
    original_lines = original_content.splitlines(keepends=True)
    patched_lines = patched_content.splitlines(keepends=True)
    
    diff = difflib.unified_diff(
        original_lines,
        patched_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
        n=3
    )
    
    diff_lines = list(diff)
    if diff_lines:
        git_header = f"diff --git a/{file_path} b/{file_path}\n"
        return git_header + "".join(diff_lines)
    return ""

def try_parse_json(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

## 5. Main Pipeline

### A. Integrate with SWE-Bench Evaluation 

In [41]:
def normalize_model_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()

    # Strip fenced block
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_+-]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Extract inner fenced block 
    m = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # Remove common chatty prefixes
    text = re.sub(
        r"^\s*(here(?:'s| is)\s+the\s+(?:fixed|corrected)\s+(?:function|code)\s*:?)",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    return text

def preserve_indentation(original_func: str, patched_func: str) -> str:
    # Detect base indentation (original)
    orig_lines = original_func.splitlines()
    orig_indent = ""
    for line in orig_lines:
        if line.strip():
            orig_indent = line[: len(line) - len(line.lstrip())]
            break

    # Detect base indentation (patch)
    patch_lines = patched_func.splitlines()
    patch_indent = ""
    for line in patch_lines:
        if line.strip():
            patch_indent = line[: len(line) - len(line.lstrip())]
            break

    # Re-add lost indentation
    if orig_indent and not patch_indent:
        patched_func = "\n".join(orig_indent + line if line.strip() else line
                                  for line in patch_lines)

    return patched_func


def returned_function_name(code: str) -> Optional[str]:
    m = re.search(r"^\s*def\s+([A-Za-z_]\w*)\s*\(", code, flags=re.MULTILINE)
    return m.group(1) if m else None


def run_msr_on_swebench(
    instance: Dict[str, Any],
    repo_path: Path
) -> Optional[str]:
    instance_id = instance.get("instance_id", "unknown")
    print(f"\\n--- Processing {instance_id} ---")
 
    problem = instance.get("problem_statement", "")
    if not problem:
        print("No problem statement found.")
        return None
 
    localized = localize_instance(
        instance=instance,
        repo_path=repo_path,
        run_llm_fn=run_groq,             
        file_model="llama-3.3-70b-versatile",
        func_model="llama-3.3-70b-versatile",
        max_files=5,
        max_funcs_per_file=3,
    )
 
    all_patches = []
    best_score = -float("inf")
    best_patch = None
    best_patch_info = {}
 
    for file_info in localized:
        file_path_str = file_info["file"]
        content = file_info["content"]
        functions = file_info["functions"]
 
        print(f"\\nProcessing file: {file_path_str} ({len(functions)} target functions)")
 
        # Build retrieved_context string for the repair prompt
        retrieved_context = f"# TARGET FILE: {file_path_str}\\n{content[:8000]}"
 
        for func in functions:
            func_name = func.get("name", "unknown")
            func_content = func.get("content", "")
 
            if not func_content.strip():
                continue
 
            # Build repair prompt
            prompt = build_repair_prompt_swe(
                vuln_func=func_content,
                desc=problem,
                func_name=func_name,
                retrieved_context=retrieved_context,
            )
 
            try:
                raw_patched_func = run_groq(prompt)
            except Exception as e:
                print(f"    API error: {e}")
                continue
 
            patched_func = normalize_model_output(raw_patched_func)
 
            if not patched_func or patched_func.strip() == func_content.strip():
                continue
 
            patched_name = returned_function_name(patched_func)
            if not patched_name or patched_name != func_name:
                continue
 
            # Heuristics 
            heuristics = quick_heuristics(func_content, patched_func)
            if (
                not heuristics.get("non_empty_patch")
                or heuristics.get("same_text")
                or not heuristics.get("syntax_valid")
                or heuristics.get("has_markdown_fences")
                or heuristics.get("has_xml_tags")
                or heuristics.get("has_chatty_prefix")
                or heuristics.get("too_large")
            ):
                continue
 
            # LLM eval 
            try:
                eval_result = evaluate_patch_swe(
                    original_func=func_content,
                    patched_func=patched_func,
                    problem_desc=problem,
                    target_func_name=func_name,
                    model="llama-3.1-8b-instant",
                )
            except Exception as e:
                print(f"    Evaluation error: {e}")
                continue
 
            score = eval_result.get("final_score", float(heuristics.get("score_heuristic", 0.0)))
            verdict = (eval_result.get("llm_eval") or {}).get("verdict", "N/A")
 
            # Reconstruct file + validate 
            try:
                patched_func_for_file = preserve_indentation(func["content"], patched_func)
                patched_content = replace_function_in_file(
                    content, func_content, patched_func_for_file, func
                )
                ast.parse(patched_content)
                diff = create_diff_from_replacement(content, patched_content, file_path_str)
                if not diff:
                    continue
            except Exception:
                continue
 
            patch_info = {
                "diff": diff,
                "score": score,
                "eval": eval_result,
                "file": file_path_str,
                "function": func_name,
                "heuristics": heuristics,
            }
            all_patches.append(patch_info)
 
            if score > best_score:
                best_score = score
                best_patch = diff
                best_patch_info = patch_info
                print(f"    ✓ NEW BEST PATCH! Score: {score:.2f} | Verdict: {verdict}")
 
    if best_patch:
        print(f"\\n✓ Best patch: {best_patch_info.get('file')} - {best_patch_info.get('function')} (score={best_score:.2f})")
        with open(ARTIFACT_DIR / f"patch_metadata_{instance_id}.json", "w", encoding="utf-8") as f:
            json.dump({
                "instance_id": instance_id,
                "best_score": best_score,
                "best_patch_info": {k: v for k, v in best_patch_info.items() if k != "diff"},
                "all_patches_count": len(all_patches),
            }, f, indent=2)
        return best_patch
 
    print(f"\\n✗ No valid patches generated for {instance_id}")
    return None

## 6. Testing

### A. One Instance

In [65]:
def test_one_swebench_instance(
    dataset_name: str = "princeton-nlp/SWE-bench_Lite",
    split: str = "test",
    index: int = 0,
    save_prediction: bool = True,
    output_file: str = "msr_on_swebench_predictions_test(1 instance).jsonl"
):
    ds = load_dataset(dataset_name, split=split)

    if index < 0 or index >= len(ds):
        raise IndexError(f"Index {index} out of range for split of size {len(ds)}")

    instance = ds[index]

    print("=" * 80)
    print(f"Testing single instance at index {index}")
    print(f"instance_id   : {instance['instance_id']}")
    print(f"repo          : {instance['repo']}")
    print(f"base_commit   : {instance['base_commit']}")
    print("=" * 80)

    # Checkout repo
    repo_path = checkout_repo(instance["repo"], instance["base_commit"])

    # Function-level pipeline
    patch = run_msr_on_swebench(instance, repo_path)

    if patch:
        pred = {
            "instance_id": instance["instance_id"],
            "model_patch": patch,
            "model_name_or_path": "msr-groq-llama3.3-70b-versatile"
        }

        print("\n✓ Patch generated successfully")
        print(f"Patch length: {len(patch)} chars")

        if save_prediction:
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(json.dumps(pred) + "\n")
            print(f"Saved test prediction to {output_file}")

        return pred
    else:
        print("\n✗ No patch generated for this instance")
        return None

In [67]:
one_result = test_one_swebench_instance(index=0)

Testing single instance at index 0
instance_id   : astropy__astropy-12907
repo          : astropy/astropy
base_commit   : d16bfe05a744909de4b27f5875fe0d4ed41ce607
[fetch] astropy/astropy
[checkout] astropy/astropy @ d16bfe05a744909de4b27f5875fe0d4ed41ce607

--- Processing astropy__astropy-12907 ---
[fetch] astropy/astropy
[checkout] astropy/astropy @ d16bfe05a744909de4b27f5875fe0d4ed41ce607

Processing file: astropy/modeling/core.py
    ✓ NEW BEST PATCH! Score: 9.57
    ✓ NEW BEST PATCH! Score: 9.80

Processing file: astropy/extern/jquery/data/js/jquery.dataTables.js
Syntax error parsing file: invalid character '©' (U+00A9) (<unknown>, line 2)

Processing file: astropy/modeling/fitting.py
Prompt too large, truncating...
Prompt too large, truncating...
Prompt too large, truncating...

Processing file: astropy/modeling/functional_models.py

Processing file: astropy/modeling/separable.py

Processing file: astropy/wcs/wcs.py
Prompt too large, truncating...

✓ Best patch score: 9.80 from as

#### B. Run Small Batch Test

In [37]:
def test_small_batch_swebench(
    dataset_name: str = "princeton-nlp/SWE-bench_Lite",
    split: str = "test",
    start_index: int = 0,
    limit: int = 3,
    output_file: str = "msr_on_swebench_predictions_smallbatch(50).jsonl"
):
    ds = load_dataset(dataset_name, split=split)
    predictions = []

    # Reset output file for clean testing
    with open(output_file, "w", encoding="utf-8") as f:
        pass

    end_index = min(start_index + limit, len(ds))

    for i in range(start_index, end_index):
        instance = ds[i]

        print("\n" + "=" * 80)
        print(f"Processing {i - start_index + 1}/{limit}")
        print(f"Global index : {i}")
        print(f"instance_id  : {instance['instance_id']}")
        print("=" * 80)

        try:
            repo_path = checkout_repo(instance["repo"], instance["base_commit"])
            patch = run_msr_on_swebench(instance, repo_path)

            if patch:
                pred = {
                "instance_id": instance["instance_id"],
                "model_patch": patch,
                "model_name_or_path": "msr-groq-llama3.3-70b-versatile"
                }
                predictions.append(pred)

                with open(output_file, "a", encoding="utf-8") as f:
                    f.write(json.dumps(pred) + "\n")

                print("✓ Saved prediction")
            else:
                print("✗ No patch generated")

        except Exception as e:
            print(f"Error on instance {instance['instance_id']}: {e}")

        time.sleep(1)

    print(f"\nDone. Generated {len(predictions)} predictions out of {limit} attempted.")
    return predictions

In [38]:
small_results = test_small_batch_swebench(start_index=0, limit=50)


Processing 1/50
Global index : 0
instance_id  : astropy__astropy-12907
[fetch] astropy/astropy
[checkout] astropy/astropy @ d16bfe05a744909de4b27f5875fe0d4ed41ce607
\n--- Processing astropy__astropy-12907 ---
  [localize] Stage 1: asking LLM to pick files...
  [localize] LLM selected files: ['astropy/modeling/separable.py', 'astropy/io/misc/asdf/tags/transform/compound.py', 'astropy/modeling/core.py', 'astropy/modeling/models.py', 'astropy/modeling/utils.py']
  [localize] Stage 2: asking LLM to pick functions in astropy/modeling/separable.py...
  [localize] Target functions in astropy/modeling/separable.py: ['separability_matrix', 'is_separable', '_compute_n_outputs']
  [localize] Stage 2: asking LLM to pick functions in astropy/io/misc/asdf/tags/transform/compound.py...
  [localize] Target functions in astropy/io/misc/asdf/tags/transform/compound.py: ['to_tree_transform', 'from_tree_transform', 'assert_equal']
  [localize] Stage 2: asking LLM to pick functions in astropy/modeling/cor

In [63]:
process_swebench_with_msr(
    start_index=50, limit=100,
    output_file="msr_on_swebench_predictions_300.jsonl",
    overwrite_output=True
)


Processing 1/100
Global index : 50
instance_id  : django__django-13230
[fetch] django/django
[checkout] django/django @ 184a6eebb0ef56d5f1b1315a8e666830e37f3f81
\n--- Processing django__django-13230 ---
  [localize] Stage 1: asking LLM to pick files...
  [localize] LLM selected files: ['django/contrib/syndication/views.py', 'django/utils/feedgenerator.py', 'django/contrib/syndication/apps.py', 'django/core/files/utils.py', 'django/views/__init__.py']
  [localize] Stage 2: asking LLM to pick functions in django/contrib/syndication/views.py...
  [localize] Target functions in django/contrib/syndication/views.py: ['feed_extra_kwargs', 'item_extra_kwargs', 'get_feed']
  [localize] Stage 2: asking LLM to pick functions in django/utils/feedgenerator.py...
  [localize] Target functions in django/utils/feedgenerator.py: ['add_item', 'item_attributes', 'add_item_elements']
  [localize] Stage 2: asking LLM to pick functions in django/contrib/syndication/apps.py...
  [localize] No functions sele

[{'instance_id': 'django__django-13230',
  'model_patch': 'diff --git a/django/contrib/syndication/views.py b/django/contrib/syndication/views.py\n--- a/django/contrib/syndication/views.py\n+++ b/django/contrib/syndication/views.py\n@@ -98,7 +98,7 @@\n         Return an extra keyword arguments dictionary that is used when\n         initializing the feed generator.\n         """\n-        return {}\n+        return {\'comments\': self._get_dynamic_attr(\'item_comments\', obj)}\n \n     def item_extra_kwargs(self, item):\n         """\n',
  'model_name_or_path': 'msr-groq-llama3.3-70b-versatile'},
 {'instance_id': 'django__django-13265',
  'model_patch': "diff --git a/django/db/models/options.py b/django/db/models/options.py\n--- a/django/db/models/options.py\n+++ b/django/db/models/options.py\n@@ -176,6 +176,12 @@\n                     setattr(self, attr_name, getattr(self.meta, attr_name))\n                     self.original_attrs[attr_name] = getattr(self, attr_name)\n \n+            

In [64]:
process_swebench_with_msr(
    start_index=150, limit=100,
    output_file="msr_on_swebench_predictions_600.jsonl",
    overwrite_output=False
)


Processing 1/100
Global index : 150
instance_id  : psf__requests-1963
[clone] https://github.com/psf/requests.git
[fetch] psf/requests
[checkout] psf/requests @ 110048f9837f8441ea536804115e80b69f400277
\n--- Processing psf__requests-1963 ---
  [localize] Stage 1: asking LLM to pick files...
  [localize] LLM selected files: ['requests/sessions.py', 'requests/adapters.py', 'requests/models.py', 'requests/hooks.py', 'requests/utils.py']
  [localize] Stage 2: asking LLM to pick functions in requests/sessions.py...
  [localize] Target functions in requests/sessions.py: ['resolve_redirects', 'prepare_request', 'send']
  [localize] Stage 2: asking LLM to pick functions in requests/adapters.py...
  [localize] Target functions in requests/adapters.py: ['send', 'build_response', 'request_url']
  [localize] Stage 2: asking LLM to pick functions in requests/models.py...
  [localize] Target functions in requests/models.py: ['copy', 'prepare_method']
  [localize] Stage 2: asking LLM to pick functio

[{'instance_id': 'psf__requests-1963',
  'model_patch': 'diff --git a/requests/sessions.py b/requests/sessions.py\n--- a/requests/sessions.py\n+++ b/requests/sessions.py\n@@ -82,106 +82,106 @@\n \n class SessionRedirectMixin(object):\n     def resolve_redirects(self, resp, req, stream=False, timeout=None,\n-                          verify=True, cert=None, proxies=None):\n-        """Receives a Response. Returns a generator of Responses."""\n-\n-        i = 0\n-\n-        while resp.is_redirect:\n-            prepared_request = req.copy()\n-\n-            resp.content  # Consume socket so it can be released\n-\n-            if i >= self.max_redirects:\n-                raise TooManyRedirects(\'Exceeded %s redirects.\' % self.max_redirects)\n-\n-            # Release the connection back into the pool.\n-            resp.close()\n-\n-            url = resp.headers[\'location\']\n-            method = req.method\n-\n-            # Handle redirection without scheme (see: RFC 1808 Section 4

In [65]:
process_swebench_with_msr(
    start_index=250, limit=50,
    output_file="msr_on_swebench_predictions_900.jsonl",
    overwrite_output=False
)


Processing 1/50
Global index : 250
instance_id  : sympy__sympy-15308
[fetch] sympy/sympy
[checkout] sympy/sympy @ fb59d703e6863ed803c98177b59197b5513332e9
\n--- Processing sympy__sympy-15308 ---
  [localize] Stage 1: asking LLM to pick files...
  [localize] LLM selected files: ['sympy/interactive/printing.py', 'sympy/physics/vector/printing.py', 'sympy/matrices/__init__.py', 'sympy/matrices/matrices.py', 'sympy/printing/latex.py']
  [localize] Stage 2: asking LLM to pick functions in sympy/interactive/printing.py...
  [localize] Target functions in sympy/interactive/printing.py: ['_can_print_latex', '_print_latex_text', '_result_display']
  [localize] Stage 2: asking LLM to pick functions in sympy/physics/vector/printing.py...
  [localize] Target functions in sympy/physics/vector/printing.py: ['vlatex', '_print_Function']
  [localize] Stage 2: asking LLM to pick functions in sympy/matrices/__init__.py...
  [localize] No functions selected in sympy/matrices/__init__.py, skipping.
  [lo

[{'instance_id': 'sympy__sympy-15308',
  'model_patch': 'diff --git a/sympy/interactive/printing.py b/sympy/interactive/printing.py\n--- a/sympy/interactive/printing.py\n+++ b/sympy/interactive/printing.py\n@@ -166,6 +166,8 @@\n         """\n         if _can_print_latex(o):\n             s = latex(o, mode=\'plain\', **settings)\n+            if \'Trace\' in s:\n+                s = s.replace(\'Trace\', r\'\\operatorname{Tr}\')\n             s = s.strip(\'$\')\n             return \'$$%s$$\' % s\n \n',
  'model_name_or_path': 'msr-groq-llama3.3-70b-versatile'},
 {'instance_id': 'sympy__sympy-15345',
  'model_patch': "diff --git a/sympy/parsing/mathematica.py b/sympy/parsing/mathematica.py\n--- a/sympy/parsing/mathematica.py\n+++ b/sympy/parsing/mathematica.py\n@@ -15,7 +15,10 @@\n     variable-length argument needs '*' character '''\n \n     parser = MathematicaParser(additional_translations)\n-    return sympify(parser.parse(s))\n+    s = parser.parse(s)\n+    if 'Max' in s:\n+        

#### Full Processing Helper

In [43]:
def process_swebench_with_msr(
    dataset_name: str = "princeton-nlp/SWE-bench_Lite",
    split: str = "test",
    limit: Optional[int] = 10,
    start_index: int = 0,
    output_file: str = "msr_on_swebench_predictions_full_LLM.jsonl",
    overwrite_output: bool = True
):

    ds = load_dataset(dataset_name, split=split)
    predictions = []

    total = len(ds)
    end_index = total if limit is None else min(start_index + limit, total)

    if overwrite_output:
        with open(output_file, "w", encoding="utf-8") as f:
            pass

    for i in range(start_index, end_index):
        instance = ds[i]

        print("\n" + "=" * 80)
        print(f"Processing {i - start_index + 1}/{end_index - start_index}")
        print(f"Global index : {i}")
        print(f"instance_id  : {instance['instance_id']}")
        print("=" * 80)

        try:
            # Checkout repo
            repo_path = checkout_repo(instance["repo"], instance["base_commit"])

            # Run function-level MSR pipeline
            patch = run_msr_on_swebench(instance, repo_path)

            if patch:
                pred = {
                    "instance_id": instance["instance_id"],
                    "model_patch": patch,
                    "model_name_or_path": "msr-groq-llama3.3-70b-versatile"
                }
                predictions.append(pred)

                # Save incrementally
                with open(output_file, "a", encoding="utf-8") as f:
                    f.write(json.dumps(pred) + "\n")

                print("✓ Prediction saved")
            else:
                print("✗ No valid patch generated")

        except Exception as e:
            print(f"Error processing {instance['instance_id']}: {e}")

        time.sleep(1)  

    print(f"\nFinished. Generated {len(predictions)} predictions.")
    print(f"Saved to: {output_file}")
    return predictions

In [45]:
all_results = process_swebench_with_msr(
    limit=None,
    output_file="msr_on_swebench_predictions_full_LLM.jsonl"
)


Processing 1/300
Global index : 0
instance_id  : astropy__astropy-12907
[fetch] astropy/astropy
[checkout] astropy/astropy @ d16bfe05a744909de4b27f5875fe0d4ed41ce607
\n--- Processing astropy__astropy-12907 ---
  [localize] Stage 1: asking LLM to pick files...
  [localize] LLM selected files: ['astropy/modeling/separable.py', 'astropy/io/misc/asdf/tags/transform/compound.py', 'astropy/modeling/core.py', 'astropy/modeling/models.py', 'astropy/modeling/utils.py']
  [localize] Stage 2: asking LLM to pick functions in astropy/modeling/separable.py...
  [localize] Target functions in astropy/modeling/separable.py: ['separability_matrix', 'is_separable', '_compute_n_outputs']
  [localize] Stage 2: asking LLM to pick functions in astropy/io/misc/asdf/tags/transform/compound.py...
  [localize] Target functions in astropy/io/misc/asdf/tags/transform/compound.py: ['from_tree_transform', 'to_tree_transform']
  [localize] Stage 2: asking LLM to pick functions in astropy/modeling/core.py...
  [loca